# Final analysis: expected-return decisions, reviewer breakeven, realistic label arrival, deployment-realistic training
All **real** Lending Club loans (36-month, issued 2007-2015). Run All; roughly 10-30 minutes. Sections are isolated (a failure does not stop the rest); the last cell lists failures and zips everything to `Desktop/Q2/outputs_final.zip` - send me that zip.

**Needs** in `Desktop/Q2`: `extra_data/lending_club_36m_wide.csv`, `extra_data/payments.csv` (created by the previous notebook) and the raw Kaggle file in `archive/` (read once more for `last_pymnt_d`, cached as `extra_data/label_dates.csv`).

| # | section | question |
|---|---|---|
| A | profit frontier | for every approval share, how much realised profit does each *ranking* earn (default model, expected-return model, the lender's own rate/grade, hindsight)? |
| B | expected-return model + conformal lower bound | does modelling profit per dollar directly beat modelling default; is a conformal one-sided lower bound on return valid under shift? |
| C | reviewer breakeven | how accurate must a human reviewer be for conformal routing to beat approving everyone? |
| D | realistic label arrival | defaults are observed earlier than repayments: how big is the resulting bias, and what does it do to adaptive conformal? |
| E | deployment-realistic training | the paper's split trains on 2011 loans whose outcomes were not all known on 1 Jan 2013. What changes when only labels known by then are used? |
| F | paper tables | data summary by year |
| G | analytical-property checks | Section 3.6 of the paper on the loan-level predictions: ties and A1, the exact ranking cut (Proposition 5), 1,000-repeat random halves (Proposition 4), the 2,047-loan calibration-size experiment with an exchangeable benchmark, and a direct estimate of kappa (Proposition 2) |

**Approximation to state in the paper:** the raw file has the date of the last payment, not the charge-off date. A label is taken as known at the last-payment month for repaid loans and at the last-payment month **+ a charge-off lag (4 months, with 2 and 6 as sensitivity)** for defaults.

In [ ]:
import os, sys, subprocess, importlib, warnings, json, gzip, zipfile, hashlib, time, traceback, contextlib, shutil
from pathlib import Path
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('lightgbm'); ensure('statsmodels'); ensure('scikit-learn', 'sklearn')
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib
matplotlib.use(os.environ.get('MPL_BACKEND', matplotlib.get_backend()))
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr
warnings.filterwarnings('ignore'); plt.style.use('seaborn-v0_8-whitegrid')
def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2(); assert Q2 is not None, "No 'Q2' folder on the Desktop; set Q2 = Path(r'...') here."
OUT = Q2/'outputs_final'; TAB, FIG = OUT/'tables', OUT/'figures'; TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
SEED, ALPHA = 42, 0.10
MAX_PER_MONTH = int(os.environ.get('MAX_PER_MONTH', 2500))      # only for the row-by-row online methods in section D
CHARGEOFF_LAG = 4; LAGS = (2, 4, 6)
FAILED = []; T0 = time.time()
@contextlib.contextmanager
def section(name):
    t = time.time(); print(f"\n{'=' * 10} {name} {'=' * 10}")
    try: yield
    except Exception:
        FAILED.append(name); print(f"!!! SECTION FAILED: {name}"); traceback.print_exc()
    finally: print(f"[{name}: {time.time() - t:.0f}s | total {(time.time() - T0) / 60:.1f} min]")
def save(df, name): df.round(5).to_csv(TAB/f'{name}.csv', index=False)
def savefig(name):
    for ext in ('png', 'pdf'): plt.savefig(FIG/f'{name}.{ext}', dpi=180, bbox_inches='tight')
def conf_q(scores, alpha):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]
def true_scores(p, y): return np.where(y == 1, 1 - p, p)
def conformal_sets(p, y, ca, alpha=ALPHA, nbin=5):
    sc = true_scores(p, y); q = conf_q(sc[ca], alpha); q0, q1 = conf_q(sc[ca & (y == 0)], alpha), conf_q(sc[ca & (y == 1)], alpha)
    edges = np.quantile(p[ca], np.linspace(0, 1, nbin + 1)[1:-1]); b = np.digitize(p, edges)
    qb = np.array([conf_q(sc[ca & (b == k)], alpha) for k in range(nbin)])[b]
    return {'static': np.column_stack([p <= q, (1 - p) <= q]), 'class-conditional': np.column_stack([p <= q0, (1 - p) <= q1]),
            'bin-conditional': np.column_stack([p <= qb, (1 - p) <= qb])}
def cov_stats(S, y):
    hit = S[np.arange(len(y)), y]; ap, rj = S[:, 0] & ~S[:, 1], S[:, 1] & ~S[:, 0]
    return dict(cov=hit.mean(), cov_default=hit[y == 1].mean(), size=S.sum(1).mean(), share_review=1 - ap.mean() - rj.mean(), share_reject_only=rj.mean())
def tiers(S): ap = S[:, 0] & ~S[:, 1]; rj = S[:, 1] & ~S[:, 0]; return ap, ~(ap | rj), rj
print("Q2:", Q2)

In [ ]:
# ---------- data, payments, label dates, models ----------
with section('0. data and models'):
    W = pd.read_csv(Q2/'extra_data'/'lending_club_36m_wide.csv', dtype={'id': str})
    def find_raw():
        arc = Q2/'archive'; files = [p for p in arc.rglob('*') if p.is_file()] if arc.is_dir() else []
        acc = [p for p in files if 'accepted' in p.name.lower() and p.name.lower().endswith(('.csv', '.csv.gz'))]
        if acc: return max(acc, key=lambda p: p.stat().st_size)
        for z in list(Q2.glob('*.zip')) + [p for p in files if p.suffix.lower() == '.zip']:
            with zipfile.ZipFile(z) as zf:
                hit = [n for n in zf.namelist() if 'accepted' in n.lower() and n.lower().endswith(('.csv', '.csv.gz'))]
                if hit: return (z, hit[0])
        return None
    def open_raw(raw):
        if isinstance(raw, tuple):
            f = zipfile.ZipFile(raw[0]).open(raw[1]); return gzip.open(f) if raw[1].lower().endswith('.gz') else f
        return raw
    def cached(path, cols):
        if path.exists(): return pd.read_csv(path, dtype={'id': str})
        raw = find_raw(); assert raw is not None, f"raw Kaggle file not found in Q2/archive and no cache {path.name}"
        ids = set(W['id']); parts = []
        for ch in pd.read_csv(open_raw(raw), usecols=cols, dtype={'id': str}, chunksize=250_000, low_memory=False): parts.append(ch[ch['id'].isin(ids)])
        d = pd.concat(parts).drop_duplicates('id'); d.to_csv(path, index=False); return d
    pay = cached(Q2/'extra_data'/'payments.csv', ['id', 'total_pymnt', 'funded_amnt'])[['id', 'total_pymnt']]
    lab = cached(Q2/'extra_data'/'label_dates.csv', ['id', 'last_pymnt_d'])
    W = W.merge(pay, on='id', how='left').merge(lab, on='id', how='left')
    W['profit'] = W['total_pymnt'] - W['funded_amnt']; W['roi'] = W['profit'] / W['funded_amnt']
    full = W[W['year'] <= 2015].reset_index(drop=True)
    full['grade_ord'] = full['grade'].map({g: i for i, g in enumerate('ABCDEFG')})
    full['subgrade_ord'] = full['sub_grade'].map({f"{g}{k}": 5 * i + k - 1 for i, g in enumerate('ABCDEFG') for k in range(1, 6)})
    y = full['default'].values; yr = full['year'].values; tmon = ((full['year'] - 2007) * 12 + full['month']).values
    pi, fund, roi = full['profit'].values, full['funded_amnt'].values.astype(float), full['roi'].values
    assert np.isfinite(pi).mean() > 0.99, "payments missing"
    print(f"{len(full):,} loans | mean profit/loan ${pi.mean():,.0f} | ROI {pi.sum() / fund.sum():.2%} | default {y.mean():.2%}")

    BASE_NUM = ['loan_amnt', 'annual_inc', 'dti', 'fico_score', 'revol_util', 'delinq_2yrs', 'emp_length_yrs']; BASE_CAT = ['home_ownership', 'verification_status', 'purpose']
    BUREAU = ['open_acc', 'pub_rec', 'total_acc', 'inq_last_6mths', 'mths_since_last_delinq', 'pub_rec_bankruptcies', 'revol_bal', 'credit_hist_yrs']
    FS = {'F0': (BASE_NUM + ['int_rate'], BASE_CAT), 'F3': (BASE_NUM + BUREAU, BASE_CAT + ['addr_state']),
          'F4': (BASE_NUM + BUREAU + ['int_rate', 'grade_ord', 'subgrade_ord', 'installment'], BASE_CAT + ['addr_state'])}
    Xc = {k: pd.get_dummies(full[n + c], columns=c, drop_first=True) for k, (n, c) in FS.items()}
    LGB_REG = dict(n_estimators=250, learning_rate=0.03, num_leaves=15, max_depth=4, min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10.0, random_state=SEED, verbose=-1)
    def fit_clf(kind, X, mtr):
        if kind == 'default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1)
        else: m = lgb.LGBMClassifier(**LGB_REG)
        return m.fit(X[mtr], y[mtr]).predict_proba(X)[:, 1]
    def fit_reg(X, target, mtr, quantile=None):
        m = lgb.LGBMRegressor(objective='quantile', alpha=quantile, **LGB_REG) if quantile else lgb.LGBMRegressor(**LGB_REG)
        return m.fit(X[mtr], target[mtr]).predict(X)
    TR = yr <= 2011; CA = yr == 2012; TE = yr >= 2013
    P_paper = fit_clf('default', Xc['F0'], TR); P_sel = fit_clf('reg', Xc['F3'], TR)             # same two pipelines as the previous notebook
    R_hat = {k: fit_reg(Xc[k], roi, TR) for k in ('F0', 'F3', 'F4')}                             # expected return per dollar
    blk = pd.factorize(yr[TE] * 100 + full['month'].values[TE])[0]; NB = blk.max() + 1; rng = np.random.default_rng(SEED)
    def gain_ci(contrib, B=1000):
        base = pi; a = np.bincount(blk, contrib[TE], NB); A = np.bincount(blk, base[TE], NB); n = np.bincount(blk, minlength=NB)
        ix = np.random.default_rng(SEED).integers(0, NB, (B, NB)); d = (a[ix].sum(1) - A[ix].sum(1)) / n[ix].sum(1); return np.percentile(d, [2.5, 97.5])
    print("models fitted; test loans:", TE.sum())

In [ ]:
# ---------- A. profit frontier by ranking method ----------
with section('A. profit frontier'):
    rankers = {'default prob, paper (F0 LGBM)': -P_paper, 'default prob, selected (F3 reg)': -P_sel, 'expected return, F0': R_hat['F0'],
               'expected return, F3 (applicant-only)': R_hat['F3'], 'expected return, F4 (everything)': R_hat['F4'],
               "lender's int_rate (lower = better)": -full['int_rate'].values, "lender's sub_grade (lower = better)": -full['subgrade_ord'].values.astype(float),
               'hindsight oracle (realised profit)': pi}
    idx_te = np.where(TE)[0]; N = len(idx_te); SHARES = (1.0, 0.99, 0.95, 0.90, 0.80, 0.70, 0.50); curves = {}; rows = []
    for name, s in rankers.items():
        order = idx_te[np.argsort(-s[idx_te], kind='stable')]; cum = np.cumsum(pi[order]); curves[name] = cum / N
        for sh in SHARES:
            k = int(round(sh * N)); take = np.zeros(len(y), bool); take[order[:k]] = True; contrib = np.where(take, pi, 0.0)
            lo, hi = gain_ci(contrib) if sh < 1 else (0.0, 0.0)
            rows.append(dict(ranker=name, share_approved=sh, profit_per_applicant=cum[k - 1] / N, gain_vs_approve_all=cum[k - 1] / N - pi[TE].sum() / N, gain_ci_lo=lo, gain_ci_hi=hi,
                             default_rate_approved=y[order[:k]].mean(), roi_approved=pi[order[:k]].sum() / fund[order[:k]].sum()))
    FR = pd.DataFrame(rows); save(FR, 'A1_profit_frontier'); pd.set_option('display.width', 250)
    print(FR[FR.share_approved.isin([0.95, 0.90, 0.80])].round(3).to_string(index=False))
    best = {n: (np.argmax(c) + 1) / N for n, c in curves.items()}; print("\nShare approved that maximises profit on the TEST years (hindsight):", {k: round(v, 3) for k, v in best.items()})
    # by-year gain at 90% approval (fixed share inside each year)
    yrows = []
    for name, s in rankers.items():
        for yy in (2013, 2014, 2015):
            ii = np.where(yr == yy)[0]; o = ii[np.argsort(-s[ii], kind='stable')]; k = int(0.9 * len(ii))
            yrows.append(dict(ranker=name, year=yy, gain_at_90pct_approved=(pi[o[:k]].sum() - pi[ii].sum()) / len(ii)))
    FY = pd.DataFrame(yrows); save(FY, 'A2_gain_at_90pct_by_year'); print(); print(FY.pivot(index='ranker', columns='year', values='gain_at_90pct_approved').round(1).to_string())

In [ ]:
# ---------- B. expected-return model and conformal one-sided lower bound ----------
with section('B. expected return + conformal lower bound'):
    rows = []; cq = {}
    for k in ('F0', 'F3', 'F4'):
        rh = R_hat[k]
        for yy in (2012, 2013, 2014, 2015):
            m = yr == yy; rows.append(dict(feature_set=k, year=yy, spearman=spearmanr(rh[m], roi[m]).correlation, AUC_profitable=roc_auc_score((pi[m] > 0).astype(int), rh[m]),
                                           mean_pred_roi=rh[m].mean(), mean_actual_roi=roi[m].mean(), calib_gap_pp=100 * (roi[m].mean() - rh[m].mean())))
    BQ = pd.DataFrame(rows); save(BQ, 'B1_roi_model_quality'); print(BQ.round(3).to_string(index=False))
    # CQR-style lower bound: L(x) = qhat_0.10(x) - c,  c = conformal quantile of (qhat(x_i) - roi_i) on 2012  =>  P(roi >= L) >= 0.90 if exchangeable
    LB = {}; rows = []
    for k in ('F0', 'F3'):
        qh = fit_reg(Xc[k], roi, TR, quantile=ALPHA); s = qh - roi; c = conf_q(s[CA], ALPHA); LB[k] = qh - c
        for yy in (2012, 2013, 2014, 2015):
            m = yr == yy; rows.append(dict(feature_set=k, year=yy, raw_quantile_cov=np.mean(roi[m] >= qh[m]), conformal_cov=np.mean(roi[m] >= LB[k][m]), target=1 - ALPHA, correction=c, mean_lower_bound=LB[k][m].mean()))
    BC = pd.DataFrame(rows); save(BC, 'B2_lower_bound_coverage'); print(); print(BC.round(4).to_string(index=False))
    # decisions: approve iff the 90%-lower bound on return exceeds theta
    base_all = pi[TE].sum() / TE.sum(); rows = []
    for k in ('F0', 'F3'):
        for th in (-0.60, -0.50, -0.40, -0.30, -0.20, -0.10, 0.0, 0.05):
            ap = LB[k] >= th; ap_te = ap[TE]; contrib = np.where(ap, pi, 0.0); lo, hi = gain_ci(contrib)
            rows.append(dict(feature_set=k, theta=th, share_approved=ap_te.mean(), profit_per_applicant=contrib[TE].sum() / TE.sum(), gain_vs_approve_all=contrib[TE].sum() / TE.sum() - base_all,
                             gain_ci_lo=lo, gain_ci_hi=hi, default_rate_approved=y[TE][ap_te].mean() if ap_te.any() else np.nan,
                             cov_given_approved=np.mean(roi[TE][ap_te] >= LB[k][TE][ap_te]) if ap_te.any() else np.nan))
    BD = pd.DataFrame(rows); save(BD, 'B3_decisions_on_lower_bound'); print(); print(BD.round(4).to_string(index=False))

In [ ]:
# ---------- C. reviewer-accuracy breakeven ----------
with section('C. reviewer breakeven'):
    rows = []; ca = CA
    for pname, p in (('paper: F0 + LGBM default', P_paper), ('selected: F3 + LGBM regularised', P_sel)):
        S = conformal_sets(p, y, ca)
        for lab in ('static', 'class-conditional', 'bin-conditional'):
            ap, rv, rj = tiers(S[lab]); n = TE.sum()
            def parts(ix):                                   # sums over the test loans selected by boolean array ix (aligned to full)
                return dict(rej=pi[ix & rj].sum(), pi0=pi[ix & rv & (y == 0)].sum(), pi1=pi[ix & rv & (y == 1)].sum(), fr=fund[ix & rv].sum())
            tot = parts(TE)
            for cost in (0.0, 0.01):
                qstar = (tot['rej'] + tot['pi0'] + cost * tot['fr']) / (tot['pi0'] - tot['pi1'])
                bs = []
                for b in range(1000):
                    pick = np.random.default_rng(b).integers(0, NB, NB)
                    sums = {k_: np.bincount(blk, v_, NB) for k_, v_ in (('rej', np.where(rj[TE], pi[TE], 0.)), ('pi0', np.where(rv[TE] & (y[TE] == 0), pi[TE], 0.)),
                                                                     ('pi1', np.where(rv[TE] & (y[TE] == 1), pi[TE], 0.)), ('fr', np.where(rv[TE], fund[TE], 0.)))} if b == 0 else sums
                    t_ = {k_: v_[pick].sum() for k_, v_ in sums.items()}; bs.append((t_['rej'] + t_['pi0'] + cost * t_['fr']) / (t_['pi0'] - t_['pi1']))
                row = dict(pipeline=pname, policy=lab, review_cost=cost, share_review=rv[TE].mean(), share_reject_only=rj[TE].mean(), breakeven_accuracy=qstar, ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5))
                for q in (0.6, 0.7, 0.8, 0.9, 1.0):
                    row[f'gain_at_acc_{q}'] = (-tot['rej'] - tot['pi0'] - cost * tot['fr'] + q * (tot['pi0'] - tot['pi1'])) / n
                rows.append(row)
    CB = pd.DataFrame(rows); save(CB, 'C1_reviewer_breakeven'); print(CB.round(3).to_string(index=False))
    print("\nbreakeven_accuracy = share of reviewed loans the reviewer must decide correctly (approve repayers / reject defaulters) for conformal routing to match approving everyone; 1.0 or more = never.")

In [ ]:
# ---------- D. realistic, outcome-dependent label arrival ----------
def aci_reveal(p, y_, month, reveal, cal_scores, alpha, gamma, clip=(0.01, 0.99)):
    s = np.sort(cal_scores); n = len(s); T = len(y_); a = alpha; err = np.zeros(T); size = np.zeros(T); order = np.argsort(reveal, kind='stable'); ptr = 0
    for t in range(T):
        while ptr < T and reveal[order[ptr]] <= month[t] and order[ptr] < t:
            a = np.clip(a + gamma * (alpha - err[order[ptr]]), *clip); ptr += 1
        k = int(np.ceil((n + 1) * (1 - a))); q = -np.inf if k <= 0 else (np.inf if k > n else s[k - 1])
        in0, in1 = p[t] <= q, (1 - p[t]) <= q; err[t] = 1 - (in1 if y_[t] == 1 else in0); size[t] = int(in0) + int(in1)
    return 1 - err, size
def recal_reveal(p, y_, month, reveal, is_test, pool_ok, alpha, mode, window, cal_scores, min_pool=300):
    sc = true_scores(p, y_); T = len(y_); cov = np.full(T, np.nan); size = np.full(T, np.nan)
    for m in np.unique(month[is_test]):
        idx = np.where(is_test & (month == m))[0]
        if mode == 'revealed': pool = sc[pool_ok & (reveal <= m) & (month >= m - 36)]                    # every label known by now, loans issued in the last 36 months
        else: pool = sc[pool_ok & (month <= m - 36) & (month > m - 36 - window)]                       # matured loans only (full 36-month window), last `window` issue months
        if len(pool) < min_pool: pool = cal_scores
        q = conf_q(pool, alpha); S = np.column_stack([p[idx] <= q, (1 - p[idx]) <= q]); cov[idx] = S[np.arange(len(idx)), y_[idx]]; size[idx] = S.sum(1)
    return cov[is_test], size[is_test]
LAG_RES = []; BIAS = []
with section('D. realistic label arrival'):
    ld = pd.to_datetime(full['last_pymnt_d'], format='%b-%Y', errors='coerce'); lp = ((ld.dt.year - 2007) * 12 + ld.dt.month).values.astype(float)
    miss = np.isnan(lp); print(f"loans without last_pymnt_d: {miss.sum():,} ({miss.mean():.2%}) -> fallback issue+12 (defaults) / issue+36 (repaid)")
    def reveal_for(lag):
        r = np.where(y == 1, lp + lag, lp); r = np.where(miss, tmon + np.where(y == 1, 12, 36), r); return np.maximum(r, tmon + 1)
    rv4 = reveal_for(CHARGEOFF_LAG); lagm = rv4 - tmon
    desc = pd.DataFrame({'outcome': np.where(y == 1, 'default', 'repaid'), 'lag_months': lagm}).groupby('outcome')['lag_months'].describe(percentiles=[.1, .25, .5, .75, .9]).reset_index()
    save(desc, 'D1_label_lag_by_outcome'); print("Months from issue to label known (charge-off lag = 4):"); print(desc.round(1).to_string(index=False))
    for snap, lab_ in ((84, '2013-12'), (96, '2014-12'), (108, '2015-12')):                          # label-selection bias at a snapshot
        known = (rv4 <= snap) & (tmon >= snap - 36) & (tmon <= snap); allw = (tmon >= snap - 36) & (tmon <= snap)
        BIAS.append(dict(snapshot=lab_, window='loans issued in previous 36 months', n_all=int(allw.sum()), share_labels_known=known.sum() / allw.sum(),
                         default_rate_among_known=y[known].mean(), eventual_default_rate_all=y[allw].mean()))
    BIAS = pd.DataFrame(BIAS); save(BIAS, 'D2_label_selection_bias'); print(); print(BIAS.round(4).to_string(index=False))

    # online methods on the per-month sample
    sh_ = full.sample(frac=1.0, random_state=SEED); kp = sh_[sh_.groupby(['year', 'month']).cumcount() < MAX_PER_MONTH] if MAX_PER_MONTH else sh_
    si = kp.sort_values('issue_d', kind='stable').index.values; ys, yrs_, ms = y[si], yr[si], tmon[si]; ca_s, te_s = yrs_ == 2012, yrs_ >= 2013
    NPM = pd.Series(ms[te_s]).value_counts().mean(); r1 = lambda x: float(f"{x:.1g}"); G = [r1(0.1 / NPM), r1(0.25 / NPM)]
    print(f"\nloans/month ~{NPM:.0f}; ACI step sizes (batch gain 0.1/0.25): {G}")
    for pname, pfull in (('paper: F0 + LGBM default', P_paper), ('selected: F3 + LGBM regularised', P_sel)):
        p_s = pfull[si]; cal_sc = true_scores(p_s, ys)[ca_s]; rows = []
        def add(method, lag, c_, s_):
            for per, m in (('2013', yrs_[te_s] == 2013), ('2014', yrs_[te_s] == 2014), ('2015', yrs_[te_s] == 2015), ('2013-15', np.ones(te_s.sum(), bool))):
                rows.append(dict(pipeline=pname, method=method, charge_off_lag=lag, period=per, cov=c_[m].mean(), cov_default=c_[m][ys[te_s][m] == 1].mean(), size=s_[m].mean()))
        S = conformal_sets(p_s, ys, ca_s)['static'][te_s]; add('static (2012 calibration)', '-', S[np.arange(S.shape[0]), ys[te_s]].astype(float), S.sum(1).astype(float))
        for lag in LAGS:
            rv_s = reveal_for(lag)[si]
            for g in G:
                c_, s_ = aci_reveal(p_s[te_s], ys[te_s], ms[te_s], rv_s[te_s], cal_sc, ALPHA, g); add(f'ACI gamma={g:g}, realistic arrival', lag, c_, s_)
            for mode in ('revealed', 'matured'):
                c_, s_ = recal_reveal(p_s, ys, ms, rv_s, te_s, yrs_ >= 2012, ALPHA, mode, 12, cal_sc); add(f'rolling recalibration, {mode} labels', lag, c_, s_)
        R = pd.DataFrame(rows); LAG_RES.append(R); print(f"\n{pname}"); print(R[R.period.isin(['2015', '2013-15']) & (R.charge_off_lag.astype(str).isin(['-', '4']))].round(3).to_string(index=False))
    save(pd.concat(LAG_RES), 'D3_methods_under_realistic_arrival')

In [ ]:
# ---------- E. deployment-realistic training labels ----------
with section('E. deployment-realistic training'):
    T_DEPLOY = 72                                                                                    # 2012-12: labels known when the model goes live on 1 Jan 2013
    known = (rv4 <= T_DEPLOY); rows = []
    VAR = [('V0 paper: all 2007-11 labels, all 2012 labels (outcomes not yet known in 2013)', TR, CA),
           ('V1 train on labels known by Dec-2012; calibrate on all 2012 labels', TR & known, CA),
           ('V2 train AND calibrate only on labels known by Dec-2012 (what was observable)', TR & known, CA & known)]
    for pname, kind, fsk in (('paper: F0 + LGBM default', 'default', 'F0'), ('selected: F3 + LGBM regularised', 'reg', 'F3')):
        for vname, mtr, mca in VAR:
            p = fit_clf(kind, Xc[fsk], mtr); S = conformal_sets(p, y, mca)
            row = dict(pipeline=pname, variant=vname, n_train=int(mtr.sum()), train_default_rate=y[mtr].mean(), n_calib=int(mca.sum()), calib_default_rate=y[mca].mean())
            for yy in (2013, 2014, 2015):
                m = yr == yy; cs = cov_stats(S['static'][m], y[m]); row.update({f'AUC_{yy}': roc_auc_score(y[m], p[m]), f'calib_gap_pp_{yy}': 100 * (y[m].mean() - p[m].mean()), f'static_cov_{yy}': cs['cov'], f'static_cov_default_{yy}': cs['cov_default']})
            rows.append(row)
    EV = pd.DataFrame(rows); save(EV, 'E1_deployment_realistic_training'); print(EV.round(3).T.to_string())

In [ ]:
# ---------- F. paper tables ----------
with section('F. data summary'):
    T1 = full.groupby('year').agg(N=('default', 'size'), default_rate=('default', 'mean'), mean_int_rate=('int_rate', 'mean'), mean_fico=('fico_score', 'mean'), mean_dti=('dti', 'mean'),
                                  mean_loan_amnt=('loan_amnt', 'mean'), mean_profit_per_loan=('profit', 'mean'), roi_on_funded=('profit', lambda s: s.sum() / full.loc[s.index, 'funded_amnt'].sum())).reset_index()
    T1['role'] = np.where(T1.year <= 2011, 'train', np.where(T1.year == 2012, 'calibration', 'test')); save(T1, 'F1_data_summary_by_year'); print(T1.round(3).to_string(index=False))
    # profit per $ funded by outcome, and the implied break-even default probability
    g = full.groupby('default').apply(lambda d: d['profit'].sum() / d['funded_amnt'].sum()); be = g[0] / (g[0] - g[1])
    print(f"\nprofit per $ funded: repaid {g[0]:+.3f}, default {g[1]:+.3f} -> break-even default probability ~ {be:.1%}")
    save(pd.DataFrame([dict(profit_per_dollar_repaid=g[0], profit_per_dollar_default=g[1], breakeven_default_probability=be)]), 'F2_breakeven_default_probability')

In [ ]:
# ---------- G. analytical-property checks (Section 3.6 of the paper) ----------
# Direct checks of Propositions 1, 2, 4 and 5 on the loan-level predictions (all loans, 2012 calibration).
# Tables: G1_ties_and_A1, G2_prop5_exact_cut, G3_prop4_random_halves, G4_calibration_size_with_benchmark, G5_kappa.
# All working names are local to _section_g, so later cells (figures, summary) are unaffected.
def _section_g():
    G_TIES = []; G_CUT = []; G_HALVES = []; G_SIZE = []; G_KAP = []
    def _cov_curve(cs_sorted, ts_sorted, grid):
        n = len(cs_sorted); k = np.ceil((n + 1) * (1 - grid)).astype(int)
        q = np.where(k > n, np.inf, cs_sorted[np.clip(k, 1, n) - 1])
        return np.searchsorted(ts_sorted, q, side='right') / len(ts_sorted)
    def kappa_and_abar(cs, ts, h=0.02, step=0.0025):
        cs_s, ts_s = np.sort(cs), np.sort(ts); fine = np.arange(0.01, 0.40, 0.0005)
        ab = fine[int(np.argmin(np.abs(_cov_curve(cs_s, ts_s, fine) - 0.90)))]
        def k_at(a0):
            g = np.arange(a0 - h, a0 + h + 1e-12, step); return np.polyfit(g, 1 - _cov_curve(cs_s, ts_s, g), 1)[0]
        return k_at(ALPHA), ab, k_at(max(ab, h + 0.005))
    def kappa_boot(cs, ts, B=300, seed=SEED):
        r = np.random.default_rng(seed); est = kappa_and_abar(cs, ts); bs = []
        for _ in range(B): bs.append(kappa_and_abar(r.choice(cs, len(cs)), r.choice(ts, len(ts))))
        bs = np.array(bs); return est, bs[:, 0].std(ddof=1), bs[:, 2].std(ddof=1)
    nte = TE.sum()
    for pname, p in (('paper: F0 + LGBM default', P_paper), ('selected: F3 + LGBM regularised', P_sel)):
        sc = true_scores(p, y); q = conf_q(sc[CA], ALPHA); S = conformal_sets(p, y, CA)
        # (a) ties, assumption A1, eps0 (Propositions 1, 4, 5)
        for per, m in (('2012 (calib.)', CA), ('2013-15 (test)', TE)):
            G_TIES.append(dict(pipeline=pname, period=per, N=int(m.sum()), distinct_p=int(np.unique(p[m]).size), distinct_share=np.unique(p[m]).size / m.sum(), max_p=p[m].max(),
                               share_p_above_half=(p[m] > 0.5).mean(), q_hat=q, tie_mass_at_q=np.isclose(sc[m], q).mean(), eps0=((p[m] > q) & (y[m] == 0)).sum() / max((y[m] == 0).sum(), 1)))
        # (b) Proposition 5: conformal "referrals rejected" vs the default-probability ranking cut at the same share (pooled 2013-15)
        for lab in ('static', 'class-conditional'):
            ap, rv, rj = tiers(S[lab]); k_ap = int(ap[TE].sum()); order = np.argsort(p[TE], kind='stable'); top = np.zeros(nte, bool); top[order[:k_ap]] = True
            base = pi[TE].sum() / nte; g_conf = np.where(ap[TE], pi[TE], 0).sum() / nte - base; g_rank = np.where(top, pi[TE], 0).sum() / nte - base
            cut_lo = p[TE][~ap[TE]].min()
            G_CUT.append(dict(pipeline=pname, sets=lab, approved_share=k_ap / nte, identical_sets=bool((top == ap[TE]).all()), gain_conformal=g_conf, gain_ranking_same_share=g_rank,
                              diff_dollars=g_conf - g_rank, max_p_approved=p[TE][ap[TE]].max() if k_ap else np.nan, min_p_not_approved=cut_lo, ties_at_cut=int(np.isclose(p[TE], cut_lo).sum())))
        if not pname.startswith('paper'): continue
        # (c) Proposition 4: random halves, 1000 repeats (2012 with and without a tie-breaking jitter; 2015)
        for per, yy, jit in (('2012', 2012, False), ('2012', 2012, True), ('2015', 2015, False)):
            idx = np.where(yr == yy)[0]; N = len(idx); h_ = N // 2; m_ = N - h_; k_ = int(np.ceil((h_ + 1) * (1 - ALPHA)))
            pred = np.sqrt(k_ * (h_ + 1 - k_) / ((h_ + 1) ** 2 * (h_ + 2)) + ALPHA * (1 - ALPHA) / m_)
            pj = p[idx] + (np.random.default_rng(SEED).uniform(-1e-9, 1e-9, N) if jit else 0); scj = true_scores(pj, y[idx]); covs = []
            for r in range(1000):
                perm = np.random.default_rng(10_000 + r).permutation(N); qq = conf_q(scj[perm[:h_]], ALPHA); covs.append((scj[perm[h_:]] <= qq).mean())
            sd_ = np.std(covs, ddof=1)
            G_HALVES.append(dict(period=per, N=N, jitter=jit, mean_cov=np.mean(covs), sd=sd_, predicted_sd=pred, ratio=sd_ / pred, ratio_se=1 / np.sqrt(2 * 999)))
        # (d) calibration subsample of 2,047 loans, 1000 draws: exchangeable benchmark (half A of 2012 -> fixed half B) and test years
        idx12 = np.where(CA)[0]; permb = np.random.default_rng(7).permutation(idx12); hA, hB = permb[:len(idx12) // 2], permb[len(idx12) // 2:]
        designs = [('benchmark: 2012 half A -> 2012 half B', hA, hB)] + [(str(t), idx12, np.where(yr == t)[0]) for t in (2013, 2014, 2015)]
        for lab_, pool, tst in designs:
            r = np.random.default_rng(SEED); ts_s = np.sort(sc[tst]); covs = np.empty(1000)
            for b in range(1000):
                qs = conf_q(sc[r.choice(pool, 2047, replace=False)], ALPHA); covs[b] = np.searchsorted(ts_s, qs, side='right') / len(ts_s)
            pred = np.sqrt(ALPHA * (1 - ALPHA) * (1 / 2047 - 1 / len(pool)))
            G_SIZE.append(dict(design=lab_, pool=len(pool), n_test=len(tst), sd=covs.std(ddof=1), predicted_sd_exchangeable=pred, ratio=covs.std(ddof=1) / pred, ratio_se=1 / np.sqrt(2 * 999)))
        # (e) kappa (Proposition 2): slope of test miscoverage in the working level, at alpha and at alpha-bar, bootstrap SE
        for lab_, cal_i, tst_i in [('benchmark: 2012 half A -> 2012 half B', hA, hB)] + [(str(t), idx12, np.where(yr == t)[0]) for t in (2013, 2014, 2015)] + [('pooled 2013-15', idx12, np.where(TE)[0])]:
            (k_a, ab, k_ab), se_a, se_ab = kappa_boot(sc[cal_i], sc[tst_i])
            G_KAP.append(dict(period=lab_, n_cal=len(cal_i), n_test=len(tst_i), kappa_at_alpha=k_a, kappa_at_alpha_se=se_a, alpha_bar=ab, kappa_at_alpha_bar=k_ab, kappa_at_alpha_bar_se=se_ab))
    G_TIES, G_CUT, G_HALVES, G_SIZE, G_KAP = map(pd.DataFrame, (G_TIES, G_CUT, G_HALVES, G_SIZE, G_KAP))
    save(G_TIES, 'G1_ties_and_A1'); save(G_CUT, 'G2_prop5_exact_cut'); save(G_HALVES, 'G3_prop4_random_halves'); save(G_SIZE, 'G4_calibration_size_with_benchmark'); save(G_KAP, 'G5_kappa')
    for t_, d_ in (('Ties and A1', G_TIES), ('Proposition 5: exact cut', G_CUT), ('Proposition 4: random halves (1000 repeats)', G_HALVES),
                   ('Calibration size 2,047 with exchangeable benchmark', G_SIZE), ('kappa', G_KAP)):
        print(f"\n{t_}"); print(d_.round(5).to_string(index=False))
    return G_TIES, G_CUT, G_HALVES, G_SIZE, G_KAP
G_TIES = G_CUT = G_HALVES = G_SIZE = G_KAP = pd.DataFrame()
with section('G. analytical-property checks'):
    G_TIES, G_CUT, G_HALVES, G_SIZE, G_KAP = _section_g()


In [ ]:
# ---------- figures ----------
with section('figures'):
    fig, ax = plt.subplots(1, 2, figsize=(16, 5.2)); xs = np.arange(1, N + 1) / N
    for name, c in curves.items():
        sel = xs >= 0.5; ax[0].plot(xs[sel], c[sel], label=name, lw=2.2 if 'oracle' in name else 1.6, ls='--' if 'oracle' in name else '-')
    ax[0].axhline(pi[TE].sum() / N, c='k', lw=0.8, ls=':'); ax[0].set_xlabel('share of applicants approved (best-ranked first)'); ax[0].set_ylabel('realised profit per applicant, $')
    ax[0].set_title('Profit frontier by ranking method, 2013-2015', fontweight='bold'); ax[0].legend(fontsize=7); ax[0].set_xlim(0.5, 1)
    D = FR[(FR.share_approved == 0.90) & (~FR.ranker.str.contains('oracle'))].sort_values('gain_vs_approve_all')
    ax[1].barh(D['ranker'], D['gain_vs_approve_all'], xerr=[D['gain_vs_approve_all'] - D['gain_ci_lo'], D['gain_ci_hi'] - D['gain_vs_approve_all']], color='steelblue', capsize=3)
    ax[1].axvline(0, c='k', lw=1); ax[1].set_title('Gain vs approve-all when rejecting the worst 10% (95% block CI)', fontweight='bold'); ax[1].set_xlabel('$ per applicant'); plt.tight_layout(); savefig('A1_profit_frontier'); plt.show()
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    for k, g in BC.groupby('feature_set'):
        ax[0].plot(g['year'], g['conformal_cov'], marker='o', label=f'{k} conformal'); ax[0].plot(g['year'], g['raw_quantile_cov'], marker='x', ls='--', label=f'{k} raw quantile')
    ax[0].axhline(0.9, c='k', ls=':'); ax[0].set_title('Coverage of 90% lower bound on ROI', fontweight='bold'); ax[0].set_xticks([2012, 2013, 2014, 2015]); ax[0].legend(fontsize=7)
    for (pn, cost), g in CB.groupby(['pipeline', 'review_cost']):
        if cost == 0.01: ax[1].errorbar(range(len(g)), g['breakeven_accuracy'], yerr=[g['breakeven_accuracy'] - g['ci_lo'], g['ci_hi'] - g['breakeven_accuracy']], marker='o', capsize=3, label=pn[:22])
    ax[1].set_xticks(range(3)); ax[1].set_xticklabels(['static', 'class-cond.', 'bin-cond.']); ax[1].axhline(1, c='r', ls=':'); ax[1].set_ylim(0.3, 1.1); ax[1].set_title('Reviewer accuracy needed to break even (cost 1%)', fontweight='bold'); ax[1].legend(fontsize=7)
    ax[2].bar(['default', 'repaid'], [desc.set_index('outcome').loc[o, '50%'] for o in ('default', 'repaid')], color=['darkred', 'teal']); ax[2].set_ylabel('median months from issue to label known'); ax[2].set_title('Outcome-dependent label arrival', fontweight='bold')
    plt.tight_layout(); savefig('B_C_D_summary'); plt.show()

In [ ]:
lines = [f"FINAL ANALYSIS SUMMARY ({time.strftime('%Y-%m-%d %H:%M')}, {(time.time() - T0) / 60:.1f} min)", f"Failed sections: {FAILED or 'none'}", ""]
def add_(t, d, n=80): lines.extend([f"## {t}", d.round(4).head(n).to_string(index=False), ""])
with section('summary'):
    add_('A1 frontier (95/90/80% approved)', FR[FR.share_approved.isin([0.95, 0.90, 0.80])]); add_('B2 lower-bound coverage', BC); add_('B3 decisions on lower bound', BD)
    add_('C1 reviewer breakeven', CB); add_('D1 label lag', desc); add_('D2 label-selection bias', BIAS); add_('D3 realistic arrival', pd.concat(LAG_RES)[lambda d: d.period.isin(['2015', '2013-15'])]); add_('E1 deployment-realistic training', EV); add_('G1 ties and A1', G_TIES); add_('G2 Proposition 5 exact cut', G_CUT); add_('G3 Proposition 4 random halves', G_HALVES); add_('G4 calibration size with benchmark', G_SIZE); add_('G5 kappa', G_KAP); add_('F1 data summary', T1)
(OUT/'results_summary.txt').write_text("\n".join(lines)); print("\n".join(lines)[:5000]); print(f"\nFAILED SECTIONS: {FAILED or 'none'}")
print("\nSend me this file:", shutil.make_archive(str(Q2/'outputs_final'), 'zip', OUT))